# 🛠️ Chapter 2: Pre-model Workflow and Data Preprocessing
**Referensi Buku:** *scikit-learn Cookbook, Third Edition*

---
## 1. Pendahuluan
Model *Machine Learning* sangat sensitif terhadap format data. Data berskala besar bisa mendominasi data berskala kecil, nilai kosong bisa menyebabkan error, dan teks tidak bisa diproses oleh fungsi matematika. Bab ini mengatasi semua masalah tersebut melalui *Data Preprocessing*.

In [1]:
import numpy as np
import pandas as pd
from sklearn import preprocessing

# Dataset Mentah Campuran
raw_data = pd.DataFrame({
    'Umur': [25, 30, np.nan, 45, 50],
    'Gaji': [5000, 6000, 5500, 100000, 8000],  # Perhatikan ada outlier (100.000)
    'Kota': ['Jakarta', 'Bandung', 'Jakarta', 'Surabaya', 'Bandung']
})
print("Data Mentah:\n", raw_data)

Data Mentah:
    Umur    Gaji      Kota
0  25.0    5000   Jakarta
1  30.0    6000   Bandung
2   NaN    5500   Jakarta
3  45.0  100000  Surabaya
4  50.0    8000   Bandung


## 2. Imputing Missing Values (Menangani Nilai Kosong)
Nilai `NaN` (Not a Number) harus ditangani. Kita akan menggunakan `SimpleImputer` untuk mengisinya dengan nilai median dari kolom tersebut.

In [2]:
from sklearn.impute import SimpleImputer

# Menggunakan strategy 'median'
imputer = SimpleImputer(strategy='median')

# Mengisi kolom 'Umur'
raw_data['Umur_Imputed'] = imputer.fit_transform(raw_data[['Umur']])
print("Setelah Imputasi (Perhatikan Umur yang tadinya NaN):\n", raw_data[['Umur', 'Umur_Imputed']])

Setelah Imputasi (Perhatikan Umur yang tadinya NaN):
    Umur  Umur_Imputed
0  25.0          25.0
1  30.0          30.0
2   NaN          37.5
3  45.0          45.0
4  50.0          50.0


## 3. Encoding Categorical Variables
Kolom 'Kota' berisi teks. Kita harus mengubahnya menjadi angka. Karena tidak ada urutan (Jakarta tidak lebih tinggi dari Bandung), kita menggunakan **One-Hot Encoding** agar model tidak menganggapnya sebagai ranking.

In [3]:
from sklearn.preprocessing import OneHotEncoder

# sparse=False agar output berbentuk array biasa (bukan sparse matrix)
ohe = OneHotEncoder(sparse_output=False)
kota_encoded = ohe.fit_transform(raw_data[['Kota']])

print("Kategori yang ditemukan:", ohe.categories_)
print("\nHasil One-Hot Encoding:\n", kota_encoded)

Kategori yang ditemukan: [array(['Bandung', 'Jakarta', 'Surabaya'], dtype=object)]

Hasil One-Hot Encoding:
 [[0. 1. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]


## 4. Scaling (Standardization vs RobustScaler)
Data 'Gaji' memiliki rentang ribuan, sedangkan 'Umur' hanya puluhan. Jika kita menggunakan `StandardScaler` pada data gaji, nilai *outlier* (100.000) akan merusak skala. Sebagai gantinya, buku ini memperkenalkan `RobustScaler` yang kebal terhadap outlier (berbasis kuartil).

In [4]:
from sklearn.preprocessing import StandardScaler, RobustScaler

standard_scaler = StandardScaler()
robust_scaler = RobustScaler()

gaji_standard = standard_scaler.fit_transform(raw_data[['Gaji']])
gaji_robust = robust_scaler.fit_transform(raw_data[['Gaji']])

print("Skala Standar (Terpengaruh Outlier):\n", gaji_standard.flatten())
print("\nSkala Robust (Mengamankan data normal dari Outlier):\n", gaji_robust.flatten())

Skala Standar (Terpengaruh Outlier):
 [-0.52976518 -0.50314382 -0.5164545   1.99926459 -0.44990109]

Skala Robust (Mengamankan data normal dari Outlier):
 [-0.4  0.  -0.2 37.6  0.8]


## 5. Custom Transformations
Seringkali kita perlu melakukan transformasi matematis khusus, misalnya melakukan logaritma (Log Transform) pada fitur tertentu. Kita bisa mengubah fungsi biasa menjadi Transformer resmi `scikit-learn` menggunakan `FunctionTransformer`.

In [5]:
from sklearn.preprocessing import FunctionTransformer

# Mengubah nilai array menggunakan fungsi np.log1p (Logaritma natural + 1)
log_transformer = FunctionTransformer(np.log1p, validate=True)

gaji_log = log_transformer.transform(raw_data[['Gaji']])
print("Gaji asli:\n", raw_data['Gaji'].values)
print("\nGaji setelah Log Transform (lebih mulus distribusinya):\n", gaji_log.flatten())

Gaji asli:
 [  5000   6000   5500 100000   8000]

Gaji setelah Log Transform (lebih mulus distribusinya):
 [ 8.51739317  8.6996814   8.61268517 11.51293546  8.98732181]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but FunctionTransformer was fitted without feature names
  warnings.warn(
